In [ ]:
# getting libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve,
    precision_score, recall_score, f1_score)
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')
SEED = 77

In [ ]:
# loading the data
df = pd.read_csv('../creditcard.csv')
print('shape:', df.shape)
df.head()

In [ ]:
df.info()
print('nulls:', df.isnull().sum().sum())
df.describe()

In [ ]:
# class distribution
class_counts = df['Class'].value_counts()
print(class_counts)
print(f'fraud %: {class_counts[1]/len(df)*100:.4f}%')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(['Legit', 'Fraud'], class_counts.values, color=['steelblue', 'tomato'], edgecolor='black')
axes[0].set_title('Class Count')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 500, str(v), ha='center', fontweight='bold')
axes[1].pie(class_counts.values, labels=['Legit', 'Fraud'], autopct='%1.3f%%', colors=['steelblue', 'tomato'])
axes[1].set_title('Class Proportion')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# amount distribution by class
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(df[df['Class']==0]['Amount'], bins=60, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].set_title('Legit Amounts'); axes[0].set_xlim([0, 500])
axes[1].hist(df[df['Class']==1]['Amount'], bins=40, color='tomato', alpha=0.7, edgecolor='black')
axes[1].set_title('Fraud Amounts')
plt.tight_layout()
plt.savefig('amount_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'avg legit: ${df[df["Class"]==0]["Amount"].mean():.2f}')
print(f'avg fraud: ${df[df["Class"]==1]["Amount"].mean():.2f}')

In [ ]:
# time distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(df[df['Class']==0]['Time'], bins=60, color='steelblue', alpha=0.7)
axes[0].set_title('Legit Transaction Time')
axes[1].hist(df[df['Class']==1]['Time'], bins=40, color='tomato', alpha=0.7)
axes[1].set_title('Fraud Transaction Time')
plt.tight_layout()
plt.savefig('time_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# correlation heatmap
plt.figure(figsize=(16, 10))
corr = df.drop(columns=['Time']).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0, annot=False, linewidths=0.3)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# correlation with class label
class_corr = df.corr()['Class'].drop('Class').sort_values()
plt.figure(figsize=(10, 7))
class_corr.plot(kind='barh', color=['tomato' if x < 0 else 'steelblue' for x in class_corr])
plt.title('Feature Correlation with Fraud Class')
plt.axvline(x=0, color='black', lw=0.8)
plt.tight_layout()
plt.savefig('class_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# preprocessing
df = df.drop(columns=['Time'])
scaler = StandardScaler()
df['Amount'] = scaler.fit_transform(df[['Amount']])

X = df.drop(columns=['Class'])
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)
print('train:', X_train.shape, '| test:', X_test.shape)

In [ ]:
# smote on train only
smote = SMOTE(random_state=SEED)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print('before SMOTE:', y_train.value_counts().to_dict())
print('after SMOTE: ', pd.Series(y_train_sm).value_counts().to_dict())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(['Legit', 'Fraud'], y_train.value_counts().values, color=['steelblue', 'tomato'], edgecolor='black')
axes[0].set_title('Before SMOTE')
axes[1].bar(['Legit', 'Fraud'], pd.Series(y_train_sm).value_counts().values, color=['steelblue', 'tomato'], edgecolor='black')
axes[1].set_title('After SMOTE')
plt.tight_layout()
plt.savefig('smote_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# logistic regression
lr_model = LogisticRegression(max_iter=1000, random_state=SEED)
lr_model.fit(X_train_sm, y_train_sm)
lr_preds = lr_model.predict(X_test)
lr_proba = lr_model.predict_proba(X_test)[:, 1]
print('Logistic Regression:')
print(classification_report(y_test, lr_preds, target_names=['Legit', 'Fraud']))

In [ ]:
# random forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
rf_model.fit(X_train_sm, y_train_sm)
rf_preds = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)[:, 1]
print('Random Forest:')
print(classification_report(y_test, rf_preds, target_names=['Legit', 'Fraud']))

In [ ]:
# confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, preds, title in zip(axes, [lr_preds, rf_preds], ['Logistic Regression', 'Random Forest']):
    cm = confusion_matrix(y_test, preds)
    cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
    annot = np.array([[f'{v}\n({p:.1f}%)' for v, p in zip(rv, rp)] for rv, rp in zip(cm, cm_pct)])
    sns.heatmap(cm, annot=annot, fmt='', ax=ax, cmap='Blues', linewidths=1,
                xticklabels=['Pred Legit','Pred Fraud'],
                yticklabels=['Actual Legit','Actual Fraud'], cbar=False)
    ax.set_title(f'Confusion Matrix - {title}')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# roc curves
plt.figure(figsize=(8, 6))
for proba, label, color in [(lr_proba,'Logistic Regression','steelblue'),(rf_proba,'Random Forest','tomato')]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    plt.plot(fpr, tpr, label=f'{label} (AUC={auc:.4f})', color=color, lw=2)
plt.plot([0,1],[0,1],'k--',lw=1)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curves'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# precision-recall curves
plt.figure(figsize=(8, 6))
for proba, label, color in [(lr_proba,'Logistic Regression','steelblue'),(rf_proba,'Random Forest','tomato')]:
    prec, rec, _ = precision_recall_curve(y_test, proba)
    plt.plot(rec, prec, label=label, color=color, lw=2)
plt.xlabel('Recall'); plt.ylabel('Precision')
plt.title('Precision-Recall Curves'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# feature importances
feat_imp = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False).head(15)
plt.figure(figsize=(10, 6))
feat_imp.sort_values().plot(kind='barh', color='steelblue', edgecolor='black')
plt.title('Top 15 Feature Importances - Random Forest')
plt.tight_layout()
plt.savefig('feature_importances.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# final comparison
results = {}
for name, preds, proba in [('Logistic Regression', lr_preds, lr_proba),('Random Forest', rf_preds, rf_proba)]:
    results[name] = {
        'Precision': precision_score(y_test, preds),
        'Recall':    recall_score(y_test, preds),
        'F1':        f1_score(y_test, preds),
        'ROC-AUC':   roc_auc_score(y_test, proba)
    }
results_df = pd.DataFrame(results).T.round(4)
print(results_df)

results_df.plot(kind='bar', figsize=(10, 5), rot=0, edgecolor='black',
                color=['#2196F3','#F44336','#4CAF50','#FF9800'])
plt.title('Model Comparison'); plt.ylim([0, 1.1]); plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# saving model and scaler for the flask backend
os.makedirs('../models', exist_ok=True)
joblib.dump(rf_model, '../models/rf_model.pkl')
joblib.dump(scaler,   '../models/scaler.pkl')
print('saved to /models/')